# ORBIT Phase 4 — Accelerated Colab GPU Training & Benchmark

**Objective:** Execute large-scale iterative self-play, world model fine-tuning, stabilized imagination policy training (annealed entropy, value stabilization), and 30-seed Crafter benchmark entirely on Google Colab GPU.

## 1. Environment & Architecture Specifications
- **Compute:** NVIDIA T4 / V100 / A100 GPU
- **Base World Model:** Phase 3 Promoted Champion (`results/phase3/world_model_phase3.pt`)
- **Base Policy Candidate:** Gate 6 Champion (`results/gate6/policy_h4.pt`)
- **Evaluator Invariants:** Strict visual agent boundary `act(obs_uint8, reward_prev, step)`; zero hardcoded recipe dictionaries (AST verified).

## 2. Mount Google Drive & Clone/Update Repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Clone or pull latest codebase
!git clone https://github.com/sahiltolani30/learn-a-world.git /content/learn-a-world 2>/dev/null || (cd /content/learn-a-world && git pull origin main)
%cd /content/learn-a-world

# Install dependencies
!pip install -q -r requirements.txt
!pip install -q torchvision timm matplotlib

## 3. Verify CUDA GPU Acceleration

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Device Name: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    raise SystemError('GPU not found! Please enable GPU in Runtime -> Change runtime type -> T4 GPU.')

## 4. Run Verification Unit Tests

In [ ]:
%cd /content/learn-a-world
!pytest tests/test_orbit_phase3_selfplay.py tests/test_orbit_gate6_policy.py tests/test_orbit_gate5_temporal_memory.py -v

## 5. Execute Phase 4 Experiment on GPU

Runs the full 6-stage iterative pipeline on CUDA:
1. Stage A: Self-play interaction data collection in Crafter (GPU-accelerated DINOv2 feature extraction)
2. Stage B: Multi-step world model fine-tuning on augmented replay buffer
3. Stage C & D: Held-out diagnostic evaluation & strict model freezing
4. Stage E: Actor-critic policy imagination training ($H=4$, GAE, entropy scheduling)
5. Stage F: 30-seed benchmark across Random Baseline, Gate 6 Champion, and Phase 4 policy

In [ ]:
%cd /content/learn-a-world

!python experiments/run_phase3_iterative_selfplay.py \
  --device cuda \
  --selfplay-episodes 30 \
  --wm-updates 500 \
  --policy-updates 500 \
  --eval-seeds-count 30 \
  --max-eval-steps 300 \
  --output-dir results/phase4

## 6. Display Visualizations and Benchmark Table

In [ ]:
import json
from IPython.display import Image, display

plot_path = 'results/phase4/phase3_comparison_plots.png'
display(Image(plot_path))

with open('results/phase4/phase3_experiment_metrics.json') as f:
    res = json.load(f)

print('=' * 110)
print(f"{'Policy':<25} | {'Mean Return':<14} | {'Crafter Score':<15} | {'Survival':<12} | {'Entropy':<10}")
print('-' * 110)
for key, label in [('random_baseline', 'Random Baseline'), ('gate6_champion_h4', 'Gate 6 Champion (H=4)'), ('phase3_policy', 'Phase 4 Policy')]:
    d = res[key]
    print(f"{label:<25} | {d['mean_return']:<5.2f} ± {d['std_return']:<5.2f}  | {d['crafter_score']:<14.4f}% | {d['mean_length']:<12.1f} | {d['action_entropy']:<10.3f}")
print('=' * 110)

## 7. Sync Checkpoints & Metrics to Google Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/learn-a-world/results/phase4
!cp -r results/phase4/* /content/drive/MyDrive/learn-a-world/results/phase4/
print('Phase 4 checkpoints, replay buffers, and metrics successfully backed up to Google Drive!')